In [32]:
from groq import Groq
from dotenv import load_dotenv
import os

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

GERACAO_TOOLS = """ 
Você é um assistente que ajuda a construir catálogos de ferramentas (tools) para testar sistemas de IA com function calling.

Gere exatamente 30 tools (ferramentas) fictícias, porém realistas, que representem problemas reais do dia a dia — cobrindo uma ampla variedade de domínios (produtividade, agenda, comunicação, finanças, saúde, viagens, e-commerce, desenvolvimento de software, casa inteligente, análise de dados, clima, etc.), de forma que o catálogo seja interessante e sirva como destaque em um projeto de portfólio.

Para cada tool, gere os seguintes campos:
- nome: identificador único em snake_case, em inglês, descrevendo uma ação clara (ex: schedule_meeting, get_stock_price)
- tipo: a categoria/domínio da tool (ex: "agenda", "financas", "clima")
- descricao: uma frase clara e específica do que a função faz, escrita como documentação real de uma API
- parametros: lista de parâmetros que a função recebe, cada um com: nome, tipo (string, integer, number, boolean ou array) e se é obrigatório (true/false)
- dificuldade: "facil" ou "dificil" — distribua de forma equilibrada (aproximadamente 15 fáceis e 15 difíceis). Considere "facil" quando os parâmetros são poucos e diretos; "dificil" quando há mais parâmetros, alguma ambiguidade, ou exigem inferência de contexto.

Retorne a resposta em formato JSON, como uma lista de 30 objetos, cada um seguindo exatamente essa estrutura, sem nenhum texto fora do JSON.
"""

resposta = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": GERACAO_TOOLS}],
    max_completion_tokens=4000
)

print(resposta.choices[0].message.content)


```json
[
  {
    "nome": "schedule_meeting",
    "tipo": "agenda",
    "descricao": "Agenda uma reunião com os participantes especificados.",
    "parametros": [
      {"nome": "title", "tipo": "string", "obrigatorio": true},
      {"nome": "date", "tipo": "string", "obrigatorio": true},
      {"nome": "participants", "tipo": "array", "obrigatorio": true}
    ],
    "dificuldade": "facil"
  },
  {
    "nome": "get_stock_price",
    "tipo": "financas",
    "descricao": "Retorna o preço atual de uma ação específica.",
    "parametros": [
      {"nome": "symbol", "tipo": "string", "obrigatorio": true}
    ],
    "dificuldade": "facil"
  },
  {
    "nome": "send_email",
    "tipo": "comunicacao",
    "descricao": "Envia um e-mail para o destinatário especificado.",
    "parametros": [
      {"nome": "to", "tipo": "string", "obrigatorio": true},
      {"nome": "subject", "tipo": "string", "obrigatorio": true},
      {"nome": "body", "tipo": "string", "obrigatorio": true}
    ],
    "dificu

In [33]:
from pydantic import BaseModel
from typing import Literal
import json

class Parametro(BaseModel):
    nome: str
    tipo: Literal["string", "integer", "number", "boolean", "array", "object"]
    obrigatorio: bool

class Tool(BaseModel):
    nome: str
    tipo: str
    descricao: str
    parametros: list[Parametro]
    dificuldade: Literal["facil", "dificil"]

texto = resposta.choices[0].message.content.strip()
texto = texto.removeprefix("```json").removeprefix("```").removesuffix("```").strip()

dados = json.loads(texto)
tools_catalogo = [Tool.model_validate(item) for item in dados]

print(len(tools_catalogo))

30


In [34]:
queries_geradas = {"texto_query": [],
                   "nome_tool": [],
                   "tipo_tool": [],
                   "funcao_tool": [],
                   "dificuldade_query": []}

indice_tools = {}
for cada_tool in tools_catalogo: 
    indice_tools[cada_tool.nome] = cada_tool
print(indice_tools, end="")

{'schedule_meeting': Tool(nome='schedule_meeting', tipo='agenda', descricao='Agenda uma reunião com os participantes especificados.', parametros=[Parametro(nome='title', tipo='string', obrigatorio=True), Parametro(nome='date', tipo='string', obrigatorio=True), Parametro(nome='participants', tipo='array', obrigatorio=True)], dificuldade='facil'), 'get_stock_price': Tool(nome='get_stock_price', tipo='financas', descricao='Retorna o preço atual de uma ação específica.', parametros=[Parametro(nome='symbol', tipo='string', obrigatorio=True)], dificuldade='facil'), 'send_email': Tool(nome='send_email', tipo='comunicacao', descricao='Envia um e-mail para o destinatário especificado.', parametros=[Parametro(nome='to', tipo='string', obrigatorio=True), Parametro(nome='subject', tipo='string', obrigatorio=True), Parametro(nome='body', tipo='string', obrigatorio=True)], dificuldade='facil'), 'track_package': Tool(nome='track_package', tipo='ecommerce', descricao='Rastreia o status de entrega de u

In [41]:
import time

regra_facil = "a query deve mencionar explicitamente todos os parâmetros que a tool precisa, de forma direta"
regra_dificil = "a query deve ser indireta, vaga, ou omitir algum parâmetro, forçando inferência"

def gerar_queries_para_tools(tools_subset):
    tools_texto = ""
    for texto in tools_subset:
        tools_texto += f'nome:{texto.nome}, descrição: {texto.descricao}, parâmetros: {texto.parametros}, dificuldade: {texto.dificuldade}\n'

    prompt = f"""
Você está gerando dados de treinamento para um sistema de tool use (function calling).

Existem duas categorias de dificuldade:
- "facil": {regra_facil}
- "dificil": {regra_dificil}

Abaixo está o catálogo de tools disponíveis, cada uma já marcada com sua dificuldade:
{tools_texto}

Para CADA uma dessas tools, invente exatamente 13 perguntas de usuário, em português, realistas e variadas entre si, que uma pessoa faria a um assistente e que só poderiam ser respondidas chamando aquela tool específica — seguindo a regra de dificuldade indicada para ela.

As perguntas devem soar naturais — não mencione o nome técnico da tool.

Retorne a resposta em formato JSON, como uma lista de objetos, cada um com "nome_tool" e "queries", sem nenhum texto fora do JSON.
"""
    resposta = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=4000,
    )
    texto_resp = resposta.choices[0].message.content.strip()
    texto_resp = texto_resp.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(texto_resp)

TAMANHO_LOTE = 5
dados_queries = []

for i in range(0, len(tools_catalogo), TAMANHO_LOTE):
    lote = tools_catalogo[i:i + TAMANHO_LOTE]
    print(f"Gerando queries para tools {i} a {i + len(lote) - 1}...")
    dados_queries += gerar_queries_para_tools(lote)
    time.sleep(2)

prompt_no_tool = """
Gere exatamente 100 perguntas de usuário, em português, realistas e variadas, que podem ser respondidas diretamente, SEM que o assistente precise chamar nenhuma tool (conhecimento geral, conselhos, escrita criativa, explicações, comparações, desabafos, código simples).

Retorne em formato JSON: {"nome_tool": "no_tool", "queries": [...]}, sem nenhum texto fora do JSON.
"""
resposta_nt = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt_no_tool}],
    max_completion_tokens=4000,
)
texto_nt = resposta_nt.choices[0].message.content.strip()
texto_nt = texto_nt.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
dados_queries.append(json.loads(texto_nt))

print(len(dados_queries))

Gerando queries para tools 0 a 4...
Gerando queries para tools 5 a 9...
Gerando queries para tools 10 a 14...
Gerando queries para tools 15 a 19...
Gerando queries para tools 20 a 24...
Gerando queries para tools 25 a 29...
31


In [42]:
class ToolQueries(BaseModel):
    nome_tool: str
    queries: list[str]

queries_catalogo = [ToolQueries.model_validate(item) for item in dados_queries]
print(len(queries_catalogo))

31


In [43]:
for resultado in queries_catalogo:
    if resultado.nome_tool == "no_tool":
        nome_tool, tipo_tool, funcao_tool, dificuldade = None, None, None, "sem_tool"
    else:
        tool = indice_tools[resultado.nome_tool]
        nome_tool, tipo_tool, funcao_tool, dificuldade = tool.nome, tool.tipo, tool.descricao, tool.dificuldade
        
    for pergunta in resultado.queries:
        queries_geradas["texto_query"].append(pergunta)
        queries_geradas["nome_tool"].append(nome_tool)
        queries_geradas["tipo_tool"].append(tipo_tool)
        queries_geradas["funcao_tool"].append(funcao_tool)
        queries_geradas["dificuldade_query"].append(dificuldade)

print(len(queries_geradas["texto_query"]))

518


In [44]:
import pandas as pd 

#dados das queries
df = pd.DataFrame(queries_geradas)

#salvar o arquivo
df.to_csv('dados/queries_geradas.csv', index=False)


In [ ]:
#salvar o catálogo de tools (nome, tipo, descrição, parâmetros, dificuldade
#a etapa 2 (gerar traces) precisa dessas informações pra montar o prompt de sistema
with open('dados/tools_catalogo.json', 'w', encoding='utf-8') as f:
    json.dump([tool.model_dump() for tool in tools_catalogo], f, ensure_ascii=False, indent=2)